# Web API でかな漢字変換をする (Google CGI API for Japanese Input)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/28_japanese_input/kkc_web_api.ipynb)

NLP クックブック 28-11 ／ 日本語入力（かな漢字変換）

自分で辞書やモデルを用意せずに、手軽にかな漢字変換の結果と候補を得たい。

In [1]:
import json
import urllib.parse
import urllib.request

def transliterate(hiragana):
    """文節ごとに [読み, [候補, ...]] のリストが返る"""
    url = "https://www.google.com/transliterate?" + urllib.parse.urlencode({"langpair": "ja-Hira|ja", "text": hiragana})
    with urllib.request.urlopen(url, timeout=30) as r:
        return json.loads(r.read().decode("utf-8"))

result = transliterate("きょうはいいてんきですね")
for reading, candidates in result:
    print(reading, candidates)
print("第 1 候補をつなげる:", "".join(c[0] for _, c in result))

きょうは ['今日は', 'きょうは', '教は', '強は', '凶は']
いいてんきですね ['いい天気ですね', '良い天気ですね', 'イイ天気ですね', '好い天気ですね', 'いい天気ですネ']
第 1 候補をつなげる: 今日はいい天気ですね


In [2]:
for text in ["はしのうえではしをおとした", "しぜんげんごしょりのけんきゅうをする", "きしゃのきしゃがきしゃできしゃした", "すまほでどうがをみる"]:
    r = transliterate(text)
    print("".join(c[0] for _, c in r), "|", " ".join(reading for reading, _ in r))

# 文節の区切りを指定する: 区切りたい位置にカンマを入れる
print([(reading, c[:3]) for reading, c in transliterate("ここではきものをぬぐ")])
print([(reading, c[:3]) for reading, c in transliterate("ここで,はきものを,ぬぐ")])
print([(reading, c[:3]) for reading, c in transliterate("くるまでまつ")])
print([(reading, c[:3]) for reading, c in transliterate("くる,までまつ")])

橋の上で橋を落とした | はしのうえで はしを おとした
自然言語処理の研究をする | しぜんげんごしょりの けんきゅうを する
記者の記者が記者で記者した | きしゃの きしゃが きしゃで きしゃした
スマホで動画を見る | すまほで どうがを みる
[('ここでは', ['ここでは', '個々では', '此処では']), ('きものを', ['着物を', 'きものを', 'キモノを']), ('ぬぐ', ['脱ぐ', 'ぬぐ', 'ヌグ'])]
[('ここで', ['ここで', '個々で', '此処で']), ('はきものを', ['は着物を', '履物を', '履き物を']), ('ぬぐ', ['脱ぐ', 'ぬぐ', 'ヌグ'])]
[('くるまで', ['車で', '来るまで', 'クルマで']), ('まつ', ['待つ', 'まつ', '松'])]
[('くる', ['来る', 'くる', 'クル']), ('までまつ', ['まで待つ', 'までまつ', 'マデマツ'])]


- 文節に区切ったうえで、文節ごとの候補を返す。文節の区切りが意図と違うときは、読みにカンマを入れて区切りを指定できる。
- 手軽だが、外部のサーバーに入力を送ることになる。個人情報や機密を含む文章には使わない。利用規約と呼び出しの頻度にも注意する。
- 評価用のデータを作る、自作の変換器と結果を比べる、といった用途に便利。
- ネットワークにつながっていないと使えず、応答に時間もかかるので、キー入力のたびに呼ぶ用途には向かない。